# 🧩 Integrador

**Python, Datos e Ingeniería de IA Aplicada**  
UTN Rosario — FRRO, Área de Extensión Universitaria — 2026

**Encuentro 2 · Bloque 4 — 20 minutos**

---

## Objetivo

Ensamblar en un solo script todo lo aprendido en el encuentro: tipos, funciones, archivos, JSON y APIs. No hay conceptos nuevos — el foco está en ver cómo las piezas se conectan, exportar el pipeline como script autónomo y commitear el resultado al repositorio de la cursada.

## Lo que vamos a construir

Un script que genera un **reporte semanal de campo** para una campaña de salud comunitaria:

1. Carga el registro de beneficiarias/os desde un archivo JSON
2. Consulta el pronóstico meteorológico semanal vía API
3. Identifica los días favorables para realizar visitas domiciliarias
4. Genera un reporte en texto con toda la información integrada
5. Exporta el pipeline como script `.py` ejecutable de forma autónoma
6. Guarda el reporte y lo commitea al repositorio de la cursada

---

## ◈ Preparación — imports

Todo lo que necesitamos ya lo conocemos. Lo reunimos en un solo lugar al inicio del script — que es la práctica estándar en cualquier proyecto Python.

In [ ]:
import json
import datetime
import os
import requests

print("✓ Dependencias cargadas.")
print(f"✓ Script iniciado: {datetime.datetime.now().strftime('%d/%m/%Y %H:%M')}")

---

## ◈ Funciones del script

Definimos todas las funciones antes de usarlas. Cada función hace una sola cosa y tiene un nombre que lo dice claramente.

In [ ]:
def cargar_registros(nombre_archivo):
    """Carga la lista de beneficiarios desde un archivo JSON."""
    try:
        with open(nombre_archivo, "r", encoding="utf-8") as archivo:
            registros = json.load(archivo)
        return registros

    except FileNotFoundError:
        print(f"  ✗ Archivo no encontrado: {nombre_archivo}")
        return []

    except json.JSONDecodeError:
        print(f"  ✗ El archivo {nombre_archivo} tiene formato inválido.")
        return []

print("✓ Función cargar_registros definida.")

In [ ]:
def obtener_pronostico(latitud, longitud, dias=7):
    """Consulta el pronóstico meteorológico vía Open-Meteo API."""
    variables = "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max"

    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitud}&longitude={longitud}"
        f"&daily={variables}"
        f"&timezone=America%2FArgentina%2FBuenos_Aires"
        f"&forecast_days={dias}"
    )

    try:
        respuesta = requests.get(url, timeout=10)
        respuesta.raise_for_status()
        return respuesta.json()["daily"]

    except requests.exceptions.ConnectionError:
        print("  ✗ Sin conexión. No se pudo obtener el pronóstico.")
        return None

    except Exception as error:
        print(f"  ✗ Error al consultar la API: {error}")
        return None

print("✓ Función obtener_pronostico definida.")

In [ ]:
def es_dia_favorable_para_visita(lluvia_mm, viento_kmh):
    """Evalúa si las condiciones climáticas permiten una visita domiciliaria."""
    hay_lluvia_excesiva = lluvia_mm > 5.0
    hay_viento_excesivo = viento_kmh > 35.0

    # El día es favorable solo si ninguna condición adversa se cumple
    return not hay_lluvia_excesiva and not hay_viento_excesivo

print("✓ Función es_dia_favorable_para_visita definida.")

In [ ]:
def generar_reporte(registros, pronostico, fecha_generacion):
    """Genera el texto completo del reporte semanal de campo."""
    lineas = []
    lineas.append("═" * 55)
    lineas.append("   REPORTE SEMANAL — CAMPAÑA DE SALUD COMUNITARIA")
    lineas.append(f"   Generado: {fecha_generacion.strftime('%d/%m/%Y')}")
    lineas.append("═" * 55)
    lineas.append("")

    lineas.append("── BENEFICIARIOS EN SEGUIMIENTO ACTIVO ─────────────")
    total_en_seguimiento = 0

    for registro in registros:
        if registro["en_seguimiento"]:
            total_en_seguimiento = total_en_seguimiento + 1
            nombre  = registro["nombre"]
            glucosa = registro["glucosa"]

            if glucosa > 100:
                indicador = "▲"
            else:
                indicador = "✓"

            lineas.append(f"  {indicador} {nombre:22} | Glucosa: {glucosa} mg/dL")

    lineas.append("")
    lineas.append(f"  Total en seguimiento: {total_en_seguimiento} de {len(registros)}")
    lineas.append("")

    if pronostico is not None:
        lineas.append("── PRONÓSTICO Y DISPONIBILIDAD DE VISITAS ──────────")

        fechas        = pronostico["time"]
        temp_max      = pronostico["temperature_2m_max"]
        precipitacion = pronostico["precipitation_sum"]
        viento        = pronostico["wind_speed_10m_max"]

        dias_favorables = []

        for indice in range(len(fechas)):
            fecha          = fechas[indice]
            lluvia_del_dia = precipitacion[indice]
            viento_del_dia = viento[indice]
            favorable      = es_dia_favorable_para_visita(lluvia_del_dia, viento_del_dia)

            if favorable:
                estado_visita = "✓ Apto"
                dias_favorables.append(fecha)
            else:
                estado_visita = "✗ No apto"

            lineas.append(
                f"  {fecha}  {temp_max[indice]:4.0f}°C  "
                f"Lluvia: {lluvia_del_dia:5.1f}mm  "
                f"Viento: {viento_del_dia:4.0f}km/h  {estado_visita}"
            )

        lineas.append("")

        if len(dias_favorables) > 0:
            dias_unidos = ", ".join(dias_favorables)
            lineas.append(f"  Días recomendados para visitas: {dias_unidos}")
        else:
            lineas.append("  ✗ Sin días favorables esta semana.")
    else:
        lineas.append("  Pronóstico no disponible.")

    lineas.append("")
    lineas.append("═" * 55)

    reporte_completo = "\n".join(lineas)
    return reporte_completo

print("✓ Función generar_reporte definida.")

---

## ◈ Pipeline principal

Con las funciones definidas, el pipeline principal se lee casi como lenguaje natural. Cada paso es una línea.

In [ ]:
print("── Ejecutando pipeline ──────────────────────────────")

# Paso 1: cargar los registros de beneficiarios
print("  1. Cargando registros...")
registros = cargar_registros("registros_campaña.json")
print(f"     {len(registros)} registros cargados.")

# Paso 2: consultar el pronóstico
print("  2. Consultando pronóstico...")
pronostico = obtener_pronostico(latitud=-32.9468, longitud=-60.6393, dias=7)

if pronostico is not None:
    print("     Pronóstico recibido.")
else:
    print("     Sin pronóstico. El reporte continuará sin datos climáticos.")

# Paso 3: generar el reporte
print("  3. Generando reporte...")
fecha_hoy = datetime.date.today()
reporte   = generar_reporte(registros, pronostico, fecha_hoy)

print("")
print("── Reporte generado ─────────────────────────────────")
print(reporte)

In [ ]:
# Paso 4: guardar el reporte en disco
nombre_reporte = f"reporte_{fecha_hoy.strftime('%Y-%m-%d')}.txt"

with open(nombre_reporte, "w", encoding="utf-8") as archivo:
    archivo.write(reporte)

print(f"✓ Reporte guardado: {nombre_reporte}")
print(f"✓ Pipeline completo.")

---

## ⚡ Del notebook al script autónomo

Todo lo que construimos en este notebook puede vivir en un archivo `.py` que corre solo, sin Jupyter, desde cualquier terminal.

La celda siguiente usa `%%writefile` — una instrucción especial de Jupyter que **escribe el contenido de la celda directamente a un archivo en disco**. No ejecuta el código: lo guarda como texto.

El resultado es un script listo para producción.

In [ ]:
%%writefile pipeline_campo.py
# ─────────────────────────────────────────────────────────────────────────────
# pipeline_campo.py
# Reporte semanal de campo — Campaña de Salud Comunitaria
# Exportado desde 004_integrador.ipynb
# Uso: python pipeline_campo.py
# ─────────────────────────────────────────────────────────────────────────────

import json
import datetime
import requests


def cargar_registros(nombre_archivo):
    """Carga la lista de beneficiarios desde un archivo JSON."""
    try:
        with open(nombre_archivo, "r", encoding="utf-8") as archivo:
            registros = json.load(archivo)
        return registros
    except FileNotFoundError:
        print(f"  ✗ Archivo no encontrado: {nombre_archivo}")
        return []
    except json.JSONDecodeError:
        print(f"  ✗ El archivo {nombre_archivo} tiene formato inválido.")
        return []


def obtener_pronostico(latitud, longitud, dias=7):
    """Consulta el pronóstico meteorológico vía Open-Meteo API."""
    variables = "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max"
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitud}&longitude={longitud}"
        f"&daily={variables}"
        f"&timezone=America%2FArgentina%2FBuenos_Aires"
        f"&forecast_days={dias}"
    )
    try:
        respuesta = requests.get(url, timeout=10)
        respuesta.raise_for_status()
        return respuesta.json()["daily"]
    except requests.exceptions.ConnectionError:
        print("  ✗ Sin conexión. No se pudo obtener el pronóstico.")
        return None
    except Exception as error:
        print(f"  ✗ Error al consultar la API: {error}")
        return None


def es_dia_favorable_para_visita(lluvia_mm, viento_kmh):
    """Evalúa si las condiciones climáticas permiten una visita domiciliaria."""
    hay_lluvia_excesiva = lluvia_mm > 5.0
    hay_viento_excesivo = viento_kmh > 35.0
    return not hay_lluvia_excesiva and not hay_viento_excesivo


def generar_reporte(registros, pronostico, fecha_generacion):
    """Genera el texto completo del reporte semanal de campo."""
    lineas = []
    lineas.append("═" * 55)
    lineas.append("   REPORTE SEMANAL — CAMPAÑA DE SALUD COMUNITARIA")
    lineas.append(f"   Generado: {fecha_generacion.strftime('%d/%m/%Y')}")
    lineas.append("═" * 55)
    lineas.append("")

    lineas.append("── BENEFICIARIOS EN SEGUIMIENTO ACTIVO ─────────────")
    total_en_seguimiento = 0

    for registro in registros:
        if registro["en_seguimiento"]:
            total_en_seguimiento = total_en_seguimiento + 1
            nombre  = registro["nombre"]
            glucosa = registro["glucosa"]
            indicador = "▲" if glucosa > 100 else "✓"
            lineas.append(f"  {indicador} {nombre:22} | Glucosa: {glucosa} mg/dL")

    lineas.append("")
    lineas.append(f"  Total en seguimiento: {total_en_seguimiento} de {len(registros)}")
    lineas.append("")

    if pronostico is not None:
        lineas.append("── PRONÓSTICO Y DISPONIBILIDAD DE VISITAS ──────────")
        fechas        = pronostico["time"]
        temp_max      = pronostico["temperature_2m_max"]
        precipitacion = pronostico["precipitation_sum"]
        viento        = pronostico["wind_speed_10m_max"]
        dias_favorables = []

        for indice in range(len(fechas)):
            fecha          = fechas[indice]
            lluvia_del_dia = precipitacion[indice]
            viento_del_dia = viento[indice]
            favorable      = es_dia_favorable_para_visita(lluvia_del_dia, viento_del_dia)
            estado_visita  = "✓ Apto" if favorable else "✗ No apto"
            if favorable:
                dias_favorables.append(fecha)
            lineas.append(
                f"  {fecha}  {temp_max[indice]:4.0f}°C  "
                f"Lluvia: {lluvia_del_dia:5.1f}mm  "
                f"Viento: {viento_del_dia:4.0f}km/h  {estado_visita}"
            )

        lineas.append("")
        if len(dias_favorables) > 0:
            lineas.append(f"  Días recomendados: {', '.join(dias_favorables)}")
        else:
            lineas.append("  ✗ Sin días favorables esta semana.")
    else:
        lineas.append("  Pronóstico no disponible.")

    lineas.append("")
    lineas.append("═" * 55)
    return "\n".join(lineas)


# ─── Punto de entrada ────────────────────────────────────────────────────────
# Este bloque solo corre cuando ejecutamos el archivo directamente.
# No corre si el archivo es importado como módulo desde otro script.

if __name__ == "__main__":
    print("── Ejecutando pipeline ──────────────────────────────")

    registros  = cargar_registros("registros_campaña.json")
    pronostico = obtener_pronostico(latitud=-32.9468, longitud=-60.6393, dias=7)
    fecha_hoy  = datetime.date.today()
    reporte    = generar_reporte(registros, pronostico, fecha_hoy)

    nombre_reporte = f"reporte_{fecha_hoy.strftime('%Y-%m-%d')}.txt"
    with open(nombre_reporte, "w", encoding="utf-8") as archivo:
        archivo.write(reporte)

    print(reporte)
    print(f"✓ Reporte guardado: {nombre_reporte}")

El archivo `pipeline_campo.py` ya existe en disco. Ahora lo ejecutamos como si estuviéramos en la terminal — sin abrir Jupyter, sin celdas, sin nada más que Python.

In [ ]:
# El signo ! le dice a Jupyter que ejecute este comando en la terminal del sistema
# Es exactamente lo mismo que abrir una terminal y escribir: python pipeline_campo.py
!python pipeline_campo.py

### ✎ Para pensar

- ¿Qué hace `if __name__ == "__main__"`? ¿Por qué el código dentro de ese bloque no corrió cuando definimos las funciones en el notebook?
- El script ahora puede correr desde cualquier terminal con `python pipeline_campo.py`. ¿Qué necesitaríamos para que corra **automáticamente todos los lunes a las 8am** sin que nadie lo ejecute a mano?

---

## ◈ Commit al repositorio

El trabajo está completo. Lo registramos en el repositorio de la cursada.

```bash
git status

git add 001_lenguaje.ipynb
git add 002_memoria.ipynb
git add 003_servicios.ipynb
git add 004_integrador.ipynb
git add pipeline_campo.py
git add reporte_*.txt

git commit -m "enc-02: python aplicado — lenguaje, archivos, APIs e integrador"
git push
```

**Sobre el mensaje de commit:**
- Prefijo `enc-02` para identificar el encuentro
- Descripción breve en minúsculas
- Sin punto final — es una línea de asunto, no una oración

In [ ]:
# Verificamos el estado del repo sin salir del notebook
os.system("git status")

---

## ◈ Espacio de práctica — extendé el integrador

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Elegí al menos UNA de estas extensiones:
#
# A. Agregar al reporte el promedio de glucosa del grupo en seguimiento
#    y una recomendación textual según el valor.
#
# B. Guardar también un JSON con el resumen del reporte:
#    fecha, total beneficiarios, en seguimiento, días favorables para visitas.
#
# C. Agregar el día de la semana junto a cada fecha del pronóstico.
#    Pista: datetime.date.fromisoformat(fecha).strftime("%A").
#
# D. Modificar pipeline_campo.py para aceptar el nombre del archivo JSON
#    como argumento de línea de comando.
#    Pista: import sys → sys.argv[1]
#    Uso:   python pipeline_campo.py registros_campaña.json
#
# ─────────────────────────────────────────────────────────────────────────────

---

## ⛰️ Cierre del Encuentro 2

En este encuentro construimos un programa que:

- Habla el lenguaje de Python (tipos, control, funciones)
- Lee y escribe datos que persisten en disco
- Se conecta al mundo a través de una API
- Produce un resultado útil
- **Corre de forma autónoma, sin Jupyter, como un script real**
- Lo versiona en Git

**Qué dejamos afuera a propósito:**

| Tema | Dónde aparece |
|---|---|
| Ejecutar el script automáticamente (cron, scheduler) | Encuentro 3 |
| Empaquetarlo en un contenedor Docker | Encuentro 3 |
| APIs con autenticación y headers | Encuentro 4 |
| Análisis de datos con Pandas | Encuentro 5 |
| LLMs y modelos de lenguaje | Encuentro 6 |

**Para el próximo encuentro:**

- Tener Docker instalado en la máquina
- Traer al menos una idea de proyecto integrador (dominio, problema, qué dato usarían)

---

*Encuentro 3 → Reproducibilidad, Docker y pair programming con IA*